# Notebook 3: Monte Carlo Risk and Loss

Part of *Quantifying AI Risk*. Hour 3 of 3. The final layer.

We have telemetry emitting from Hour 1. We have Bayesian scoring producing pillar posteriors and a system trust score with confidence intervals from Hour 2. Now we translate that probabilistic trust score into a probability distribution of financial outcomes, because the CFO does not care about 0.73. The CFO cares about how much to reserve, what the tail looks like, and what to tell the board.

By the end of this notebook you will have a working Monte Carlo risk engine that takes pillar posteriors, models incident probabilities and loss distributions with a correlation structure between pillars, simulates ten thousand possible financial outcomes, and produces an executive summary with expected loss, value at risk, and tail attribution.

This is the top of the stack. Each layer transforms the output below it. Engineers see telemetry. Compliance sees trust scores. Executives see dollar risk. Same evidence, three different views.

## Setup

numpy for vectorized sampling, scipy for the correlation-aware multivariate normal. If scipy is unavailable, we can do independent sampling and the main results still hold, but correlated sampling is the right default for governance risk.

In [ ]:
import random
import numpy as np
from scipy.stats import multivariate_normal, norm, lognorm

random.seed(42)
np.random.seed(42)

print("Ready.")

## Load the pillar posteriors

Notebook 2 wrote the posteriors to `data/pillar_posteriors.json`. We read from that file here, the same way a real production pipeline would. Hour 1 emits and persists. Hour 2 reads, scores, and persists. Hour 3 reads and simulates. Each stage hands off cleanly to the next.

If the file does not exist (because Notebook 2 has not been run yet), we fall back to the hardcoded posteriors that Notebook 2 produces after running the full 1000-event stream with drift injection. That keeps this notebook runnable on its own for students who want to skip ahead, while the production-style read-from-disk path is what we demonstrate first.

In [ ]:
import json
from pathlib import Path

INPUT_PATH = Path("../data/pillar_posteriors.json") if Path("../data/pillar_posteriors.json").exists() else Path("data/pillar_posteriors.json")

if INPUT_PATH.exists():
    with open(INPUT_PATH) as f:
        payload = json.load(f)
    PILLAR_POSTERIORS = payload["pillar_state"]
    print(f"Loaded pillar posteriors from {INPUT_PATH.resolve()}")
    print(f"Source: {payload['n_events']} telemetry events processed by Notebook 2")
else:
    # Fallback: hardcoded posteriors that Notebook 2 produces with the standard
    # 1000-event stream and drift injection. Same numbers, useful for skipping ahead.
    print(f"No posteriors file found at {INPUT_PATH}. Using hardcoded values from a reference run.")
    PILLAR_POSTERIORS = {
        "Fairness & Societal Impact":   {"alpha": 1586.0, "beta": 1277.0},
        "Data & Model Integrity":       {"alpha": 1102.0, "beta":  908.0},
        "Reliability & Robustness":     {"alpha":  541.0, "beta":  469.0},
        "Operational Health":           {"alpha":  375.5, "beta":  134.5},
        "Resilience & Continuity":      {"alpha":  370.5, "beta":  139.5},
    }

print("\nPillar posteriors:\n")
print(f"  {'Pillar':32s}  {'Mean':>8s}  {'Evidence':>10s}")
print(f"  {'-'*32}  {'-'*8}  {'-'*10}")
for pillar, p in PILLAR_POSTERIORS.items():
    mean = p["alpha"] / (p["alpha"] + p["beta"])
    evidence = p["alpha"] + p["beta"] - 10  # subtract prior alpha+beta = 2+8 = 10
    print(f"  {pillar:32s}  {mean:>8.3f}  {evidence:>10.0f}")

## Map posterior to incident probability

Low trust means high incident probability. The mapping is a design choice. We use a simple logistic-style function: when the posterior mean is 1.0, incident probability per month is near 0. When the posterior mean is 0.0, incident probability is high. In between, smooth transition.

The specific numbers in this function are illustrative. In production, you calibrate this against your own historical incident rate per pillar. If you do not have historical data yet, start with these defaults, log every incident that occurs, and recalibrate every quarter.

In [ ]:
def posterior_to_incident_prob(posterior_mean, max_prob=0.15, steepness=6.0):
    """
    Maps a posterior trust score to a per-month incident probability.
    Posterior mean of 1.0 -> ~0 probability. Posterior mean of 0.0 -> max_prob.
    """
    # Logistic curve centered at 0.5
    return max_prob / (1 + np.exp(steepness * (posterior_mean - 0.5)))


print("Incident probability calibration:\n")
print(f"  {'Posterior Mean':>16s}  {'Per-month Incident Prob':>24s}")
print(f"  {'-'*16}  {'-'*24}")
for trust in [0.1, 0.3, 0.5, 0.7, 0.9]:
    p = posterior_to_incident_prob(trust)
    print(f"  {trust:>16.2f}  {p:>24.4f}")

print("\nA pillar with trust of 0.3 has ~10% monthly chance of an incident.")
print("A pillar with trust of 0.9 has ~0.4% monthly chance.")

## Loss distributions per pillar

If an incident occurs, how much does it cost? Heavy-tailed. Most incidents are small and a few are catastrophic. We use lognormal distributions because they capture the heavy tail without forcing us to enumerate specific scenarios.

The parameters below are illustrative and reflect rough industry patterns: fairness incidents in regulated decisioning have high expected cost and very high tails because of class action exposure, while operational incidents are smaller and more predictable.

In [ ]:
# Lognormal parameters per pillar: (mu, sigma) of the underlying normal.
# Median loss = exp(mu). Tail heaviness grows with sigma.
LOSS_DISTRIBUTIONS = {
    "Fairness & Societal Impact":   {"mu": np.log(500_000),  "sigma": 1.8},
    "Data & Model Integrity":       {"mu": np.log(200_000),  "sigma": 1.5},
    "Reliability & Robustness":     {"mu": np.log(100_000),  "sigma": 1.3},
    "Operational Health":           {"mu": np.log( 50_000),  "sigma": 1.0},
    "Resilience & Continuity":      {"mu": np.log( 75_000),  "sigma": 1.0},
}

print("Loss distribution shapes (if incident occurs):\n")
print(f"  {'Pillar':32s}  {'Median $':>12s}  {'Mean $':>14s}  {'P95 $':>14s}  {'P99 $':>14s}")
print(f"  {'-'*32}  {'-'*12}  {'-'*14}  {'-'*14}  {'-'*14}")

for pillar, params in LOSS_DISTRIBUTIONS.items():
    mu, sigma = params["mu"], params["sigma"]
    median = np.exp(mu)
    mean = np.exp(mu + sigma**2 / 2)
    p95 = np.exp(mu + sigma * norm.ppf(0.95))
    p99 = np.exp(mu + sigma * norm.ppf(0.99))
    print(f"  {pillar:32s}  {median:>12,.0f}  {mean:>14,.0f}  {p95:>14,.0f}  {p99:>14,.0f}")

print("\nNotice the P99 is often 5-10x the median. That is the tail that")
print("classical expected-loss calculations miss.")

## Correlation between pillars

This is the part most risk models skip and it is where the real tail risk lives. In production, when one pillar fails, others usually fail with it, because they are all responding to the same underlying event. We saw this in Hour 1 when drift arrived and every signal degraded together.

We model this by sampling correlated standard normals, then transforming them back into the incident indicator and loss scales. The correlation matrix below reflects the pattern from our streaming scenario. Fairness and data integrity are strongly correlated. Operational health and resilience are strongly correlated. Cross-group correlations are moderate.

In [ ]:
PILLAR_ORDER = [
    "Fairness & Societal Impact",
    "Data & Model Integrity",
    "Reliability & Robustness",
    "Operational Health",
    "Resilience & Continuity",
]

# Correlation matrix calibrated from the Hour 1 streaming scenario.
# Illustrative. In production, compute this from your actual signal correlation.
CORRELATION_MATRIX = np.array([
    # Fair  Data  Rel   Ops   Res
    [1.00, 0.70, 0.50, 0.30, 0.30],  # Fairness
    [0.70, 1.00, 0.55, 0.35, 0.35],  # Data Integrity
    [0.50, 0.55, 1.00, 0.45, 0.45],  # Reliability
    [0.30, 0.35, 0.45, 1.00, 0.75],  # Operational Health
    [0.30, 0.35, 0.45, 0.75, 1.00],  # Resilience
])

print("Pillar correlation matrix:\n")
print(f"  {'':32s}  " + "  ".join(f"{p[:4]:>6s}" for p in PILLAR_ORDER))
for i, p in enumerate(PILLAR_ORDER):
    row = f"  {p:32s}  " + "  ".join(f"{CORRELATION_MATRIX[i][j]:>6.2f}" for j in range(5))
    print(row)

print("\nFairness and Data Integrity move together (0.70).")
print("Operational Health and Resilience move together (0.75).")
print("Independent sampling would underestimate joint failure risk.")

## Run the Monte Carlo simulation

Ten thousand simulated months. In each month, for each pillar, we sample whether an incident occurs (correlated across pillars) and if so, how much it costs (also correlated). Sum across pillars, that is the month's total loss. Repeat ten thousand times. The resulting distribution is the answer.

In [ ]:
N_SIMULATIONS = 10_000


def run_monte_carlo(pillar_posteriors, loss_distributions, correlation_matrix,
                    pillar_order, n_sims=N_SIMULATIONS):

    # Pre-compute per-pillar incident probabilities from posteriors
    incident_probs = np.array([
        posterior_to_incident_prob(
            pillar_posteriors[p]["alpha"] / (pillar_posteriors[p]["alpha"] + pillar_posteriors[p]["beta"])
        )
        for p in pillar_order
    ])

    # Thresholds on the standard normal that correspond to those probabilities
    # If correlated-normal draw > threshold, an incident occurred for that pillar
    incident_thresholds = norm.ppf(1 - incident_probs)

    # Draw correlated standard normals for incident indicators
    incident_draws = multivariate_normal.rvs(
        mean=np.zeros(len(pillar_order)),
        cov=correlation_matrix,
        size=n_sims,
    )
    incident_occurred = incident_draws > incident_thresholds  # shape (n_sims, n_pillars)

    # For each pillar, draw lognormal losses (also correlated, using same matrix)
    loss_draws = multivariate_normal.rvs(
        mean=np.zeros(len(pillar_order)),
        cov=correlation_matrix,
        size=n_sims,
    )

    losses = np.zeros_like(incident_occurred, dtype=float)
    for j, pillar in enumerate(pillar_order):
        mu = loss_distributions[pillar]["mu"]
        sigma = loss_distributions[pillar]["sigma"]
        # Transform standard normal into lognormal loss
        losses[:, j] = np.exp(mu + sigma * loss_draws[:, j])

    # Only incurred when an incident occurred
    incurred_losses = losses * incident_occurred

    # Total per-month loss across pillars
    total_losses = incurred_losses.sum(axis=1)

    return total_losses, incurred_losses


total_losses, per_pillar_losses = run_monte_carlo(
    PILLAR_POSTERIORS, LOSS_DISTRIBUTIONS, CORRELATION_MATRIX, PILLAR_ORDER
)

print(f"Ran {N_SIMULATIONS:,} simulated months.\n")
print(f"Months with at least one incident: {(total_losses > 0).sum():,}")
print(f"Months with zero incidents:        {(total_losses == 0).sum():,}")

## The executive summary

These are the numbers that go on a slide. Expected monthly loss, value at risk at 95% and 99%, tail conditional expectation. The tail conditional expectation, which is the average loss given you are in the worst 5% of months, is the number that should keep your CFO honest, because it represents what actually happens when things go bad rather than the average case.

In [ ]:
expected_loss = total_losses.mean()
median_loss = np.median(total_losses)
p95 = np.percentile(total_losses, 95)
p99 = np.percentile(total_losses, 99)
tail_ce_95 = total_losses[total_losses >= p95].mean()
tail_ce_99 = total_losses[total_losses >= p99].mean()

print("Monthly governance risk summary\n")
print(f"  Expected loss (mean):                  ${expected_loss:>14,.0f}")
print(f"  Median loss:                           ${median_loss:>14,.0f}")
print(f"  95% VaR (95th percentile):             ${p95:>14,.0f}")
print(f"  99% VaR (99th percentile):             ${p99:>14,.0f}")
print(f"  Tail CE at 95% (avg of worst 5%):      ${tail_ce_95:>14,.0f}")
print(f"  Tail CE at 99% (avg of worst 1%):      ${tail_ce_99:>14,.0f}")

print("\nThe tail CE is higher than the VaR by design.")
print("VaR tells you where the bad zone starts. Tail CE tells you how bad it")
print("actually is once you are in the bad zone. Regulators increasingly want")
print("both because VaR alone underestimates catastrophic exposure.")

## Which pillars drive the tail?

Expected loss tells you where you are on average. Tail attribution tells you where the catastrophic risk comes from, which is often a different answer. A pillar can have low expected loss but drive a huge share of the tail, because its loss distribution is heavy and it correlates with other pillars going bad at the same time.

This is the view that turns Monte Carlo from an interesting number into a remediation plan. The pillars that drive the tail are where governance investment has the highest return on risk reduction.

In [ ]:
# Identify tail months (worst 5%)
tail_threshold = np.percentile(total_losses, 95)
tail_mask = total_losses >= tail_threshold

# Average per-pillar loss in tail months vs overall
print("Contribution to the worst 5% of months\n")
print(f"  {'Pillar':32s}  {'Avg in tail':>14s}  {'Avg overall':>14s}  {'Tail share':>12s}")
print(f"  {'-'*32}  {'-'*14}  {'-'*14}  {'-'*12}")

tail_totals = per_pillar_losses[tail_mask].mean(axis=0)
overall_totals = per_pillar_losses.mean(axis=0)
total_tail_loss = tail_totals.sum()

for j, pillar in enumerate(PILLAR_ORDER):
    tail_avg = tail_totals[j]
    overall_avg = overall_totals[j]
    share = tail_avg / total_tail_loss * 100
    print(f"  {pillar:32s}  ${tail_avg:>12,.0f}  ${overall_avg:>12,.0f}  {share:>11.1f}%")

print("\nThe pillars with the largest tail share are where governance investment")
print("reduces catastrophic exposure most. Not where average losses are highest.")

## Sensitivity: which assumption matters most?

Every number above depends on assumptions: incident probability calibration, loss distribution parameters, correlation strength. A regulator, a CFO, or an auditor will ask you which of those assumptions the result is most sensitive to. That question has a specific answer and Monte Carlo gives it to you by re-running the simulation with tweaked inputs.

In [ ]:
def sensitivity_run(tweak_fn, label):
    """Run the simulation with one tweak applied and report the delta."""
    # Copy and tweak
    posteriors = {p: dict(v) for p, v in PILLAR_POSTERIORS.items()}
    losses = {p: dict(v) for p, v in LOSS_DISTRIBUTIONS.items()}
    corr = CORRELATION_MATRIX.copy()
    tweak_fn(posteriors, losses, corr)

    totals, _ = run_monte_carlo(posteriors, losses, corr, PILLAR_ORDER, n_sims=5_000)
    new_expected = totals.mean()
    new_p99 = np.percentile(totals, 99)
    return label, new_expected, new_p99


print("Sensitivity analysis (expected loss, 99% VaR)\n")
baseline_exp = expected_loss
baseline_p99 = p99
print(f"  {'Scenario':48s}  {'Exp Loss':>12s}  {'99% VaR':>12s}")
print(f"  {'-'*48}  {'-'*12}  {'-'*12}")
print(f"  {'baseline':48s}  ${baseline_exp:>10,.0f}  ${baseline_p99:>10,.0f}")

scenarios = [
    ("fairness loss median +50%",
     lambda p, l, c: l["Fairness & Societal Impact"].update(mu=np.log(750_000))),
    ("all correlations set to 0 (naive independence)",
     lambda p, l, c: c.__setitem__(slice(None), np.eye(5))),
    ("fairness posterior improved (alpha +2000)",
     lambda p, l, c: p["Fairness & Societal Impact"].update(alpha=p["Fairness & Societal Impact"]["alpha"] + 2000)),
    ("data integrity loss tail heavier (sigma +0.5)",
     lambda p, l, c: l["Data & Model Integrity"].update(sigma=2.0)),
]

for label, tweak_fn in scenarios:
    try:
        _, exp, vap99 = sensitivity_run(tweak_fn, label)
        delta_exp_pct = (exp - baseline_exp) / baseline_exp * 100
        delta_p99_pct = (vap99 - baseline_p99) / baseline_p99 * 100
        print(f"  {label:48s}  ${exp:>10,.0f}  ${vap99:>10,.0f}   ({delta_exp_pct:+.0f}% / {delta_p99_pct:+.0f}%)")
    except Exception as e:
        print(f"  {label:48s}  error: {e}")

print("\nThe assumption the result is most sensitive to is where you invest")
print("in better measurement. An assumption the result is insensitive to can")
print("stay at default. This is the core insight of sensitivity analysis.")

## The shape of the distribution

Numbers are one thing. The shape is another. An executive who sees a histogram of monthly loss understands something the same executive staring at a single expected loss number never quite gets. The shape shows you how frequently tails actually happen, and how heavy they are.

Text histogram below. If you have matplotlib in your environment, replace with a plot.

In [ ]:
# Text histogram for environments without matplotlib
bins = [0, 50_000, 100_000, 250_000, 500_000, 1_000_000, 2_000_000, 5_000_000, float('inf')]
bin_labels = ["$0", "$0-50K", "$50-100K", "$100-250K", "$250-500K", "$500K-1M", "$1-2M", "$2-5M", "$5M+"]

# For 'zero loss' bucket, count separately
zero_count = (total_losses == 0).sum()
print("Distribution of monthly loss\n")
print(f"  {'Range':14s}  {'Count':>8s}  {'Share':>8s}  Histogram")
print(f"  {'-'*14}  {'-'*8}  {'-'*8}  {'-'*40}")
print(f"  {'No incident':14s}  {zero_count:>8,}  {zero_count/N_SIMULATIONS*100:>7.1f}%  {'#' * int(zero_count/N_SIMULATIONS*40)}")

nonzero = total_losses[total_losses > 0]
for i in range(1, len(bins)):
    lo, hi = bins[i-1], bins[i]
    count = ((nonzero > lo) & (nonzero <= hi)).sum()
    share = count / N_SIMULATIONS * 100
    bar = '#' * max(1, int(share * 2)) if count > 0 else ''
    print(f"  {bin_labels[i]:14s}  {count:>8,}  {share:>7.1f}%  {bar}")

print(f"\n  Worst month in simulation: ${total_losses.max():,.0f}")

## Your turn: reserve calculation and business decision

A Monte Carlo risk engine is only useful if the outputs drive a decision. Your task is to translate these numbers into three specific business decisions that your CFO and your AI governance committee can act on.

Your task:

1. **Reserve calculation.** Decide what reserve amount the company should hold for AI governance incidents in the coming quarter. Justify whether you use the expected loss, the 95% VaR, the 99% VaR, or something else. There is no single right answer but there is a defensible one and an indefensible one.

2. **Remediation priority.** Based on the tail attribution, rank the pillars in order of where remediation investment produces the greatest reduction in catastrophic exposure. This is different from ranking by expected loss.

3. **Telemetry investment.** Based on the sensitivity analysis, identify the one area where improving measurement (better telemetry, better calibration) would most reduce uncertainty in the risk number itself. That is where your governance engineering team should invest next.

No solution code. In production, these decisions involve judgment calls about risk tolerance, regulatory scope, and business strategy. The Monte Carlo output informs the decision but does not make it for you. Practice the judgment call now, in writing, before the stakes are real.

In [ ]:
# Your analysis space
# Use the variables already in scope:
#   expected_loss, median_loss, p95, p99, tail_ce_95, tail_ce_99
#   per_pillar_losses, tail_mask
#   PILLAR_ORDER, PILLAR_POSTERIORS, LOSS_DISTRIBUTIONS

# Print the numbers you need, then write your analysis in the markdown cell below


### Your recommendations

*Three to five sentences per item. Reserve amount and justification. Remediation priority ranking with reasoning. Top telemetry investment with expected impact.*

*This is the kind of one-page memo that goes to a governance committee or a CFO. Writing it clearly is as important as the math that produced the numbers.*

## What we just built, and what the whole stack now looks like

This notebook produced a Monte Carlo risk engine that takes pillar posteriors, models incident probabilities and loss distributions with an explicit correlation structure, runs ten thousand simulated months, and produces expected loss, value at risk, tail conditional expectation, tail attribution, sensitivity analysis, and a readable distribution view.

Zoom out. The full stack you have built across three hours:

- **Hour 1 (telemetry):** raw signals emitted from the model, structured, continuous, mapped to governance dimensions
- **Hour 2 (Bayesian scoring):** signals rolled up into pillar posteriors with confidence intervals, using a skeptical prior and severity-weighted evidence
- **Hour 3 (Monte Carlo risk):** posteriors propagated into a probability distribution of financial outcomes, with tail attribution for remediation prioritization

Each layer transforms the previous layer's output into what the next audience needs. Engineers work at Hour 1. Compliance works at Hour 2. Executives work at Hour 3. All three views share the same underlying evidence, which means the same incident affects all three views consistently. When engineers see drift in the telemetry, compliance sees the trust score drop, executives see the reserve requirement rise. Nobody is working from a different version of reality.

The specific implementation choices in these notebooks are deliberately simple so the methodology shines through. Your production deployment will have different pillars, different severity weights, different loss distributions, different correlation structure. The math stays the same. What you take back to Monday is the pattern, and the pattern is what defends itself under regulatory scrutiny.

Course complete. Now go build.